In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import sklearn as skl
# import sktime
import random
import os
# from sktime.forecasting.model_selection import temporal_train_test_split
# from sktime.utils.plotting import plot_series

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

%matplotlib inline
from matplotlib import rc
plt.rc('font', family='NanumBarunGothic')
# plt.rcParams['font.family'] ='NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(493) # Seed 고정

# 불러오기
train = pd.read_csv('./train.csv')
sales = pd.read_csv('./sales.csv')
product_info = pd.read_csv('./product_info.csv')
brand_keyword_cnt = pd.read_csv('./brand_keyword_cnt.csv')
sample_submission = pd.read_csv('./sample_submission.csv')
# price = pd.read_csv('./price.csv')

# **대분류, 중분류, 소분류, 브랜드 데이터화**

In [ ]:
base_static = train.copy()

In [ ]:
base_static.drop(['ID', '제품'], axis=1, inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

def label_encoding(data):
    label_encoder = LabelEncoder()
    categorical_columns = ['대분류', '중분류', '소분류', '브랜드']

    for col in categorical_columns:
        label_encoder.fit(data[col])
        data[col] = label_encoder.transform(data[col])

    return data

In [ ]:
label_encoding(base_static)

,대분류,중분류,소분류,브랜드,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,1,6,37,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,7,43,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,3,2,0,0,2,0
2,2,7,43,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2,7,43,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,2,7,41,3169,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15886,2,7,43,3169,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,2,4,1,1,3
15887,2,7,43,3169,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15888,2,7,43,3169,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2


In [ ]:
대분류_data = base_static.copy()
대분류_data.iloc[:, 4:] = 0
대분류_data.drop(['중분류', '소분류', '브랜드'], axis=1, inplace=True)
for i in tqdm(range(len(대분류_data))):
  대분류_data.iloc[i, 1:] = 대분류_data.iloc[i, 0]
대분류_data.drop('대분류', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
중분류_data = base_static.copy()
중분류_data.iloc[:, 4:] = 0
중분류_data.drop(['대분류', '소분류', '브랜드'], axis=1, inplace=True)
for i in tqdm(range(len(중분류_data))):
  중분류_data.iloc[i, 1:] = 중분류_data.iloc[i, 0]
중분류_data.drop('중분류', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
소분류_data = base_static.copy()
소분류_data.iloc[:, 4:] = 0
소분류_data.drop(['대분류', '중분류', '브랜드'], axis=1, inplace=True)
for i in tqdm(range(len(소분류_data))):
  소분류_data.iloc[i, 1:] = 소분류_data.iloc[i, 0]
소분류_data.drop('소분류', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
브랜드_data = base_static.copy()
브랜드_data.iloc[:, 4:] = 0
브랜드_data.drop(['대분류', '중분류', '소분류'], axis=1, inplace=True)
for i in tqdm(range(len(브랜드_data))):
  브랜드_data.iloc[i, 1:] = 브랜드_data.iloc[i, 0]
브랜드_data.drop('브랜드', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# 대분류_data.to_csv('./대분류_data.csv', index=False)
# 중분류_data.to_csv('./중분류_data.csv', index=False)
# 소분류_data.to_csv('./소분류_data.csv', index=False)
# 브랜드_data.to_csv('./브랜드_data.csv', index=False)

# **가격대**

In [ ]:
id_vars = ['브랜드']
brand_keyword_cnt_melted = pd.melt(brand_keyword_cnt, id_vars=id_vars, var_name='날짜', value_name='연관키워드_언급량')

# 날짜 컬럼을 datetime 타입으로 변환합니다.
brand_keyword_cnt_melted['날짜'] = pd.to_datetime(brand_keyword_cnt_melted['날짜'])

# 판매량을 날짜로 그룹화하고, 판매량을 합산하여 정리합니다.
brand_keyword_cnt_grouped = brand_keyword_cnt_melted.groupby(['브랜드', '날짜'])['연관키워드_언급량'].sum().reset_index()

In [ ]:
id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
train_melted = pd.melt(train, id_vars=id_vars, var_name='날짜', value_name='판매량')
train_melted['날짜'] = pd.to_datetime(train_melted['날짜'])
train_grouped = train_melted.groupby('ID')

groups = []
for i in tqdm(range(15889+1)):
    get_train = train_grouped.get_group(i).reset_index()
    get_train.drop('index', axis=1, inplace=True)
    groups.append(get_train)
train_vertical = pd.concat(groups, ignore_index=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# Function to fill the values as per the revised conditions
def fill_values(row):
    start_col = 6 # 7th column (in the actual data, this should be 6)
    fill_value = int(row[start_col]) # Convert to integer
    if fill_value == 0:
        # Find the first non-zero value after the starting column
        fill_value = int(next((value for value in row[start_col:] if value > 0), 0)) # Convert to integer
    for i in range(start_col, len(row)):
        if row[i] == 0:
            row[i] = fill_value
        elif row[i] != fill_value:
            fill_value = int(row[i]) # Convert to integer
    return row

# Apply the function to each row with tqdm
tqdm.pandas()
price_filled = price.progress_apply(fill_values, axis=1)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
price_melted = pd.melt(price_filled, id_vars=id_vars, var_name='날짜', value_name='price')
price_melted['날짜'] = pd.to_datetime(price_melted['날짜'])
price_grouped = price_melted.groupby('ID')

groups = []
for i in tqdm(range(15889+1)):
    get_price = price_grouped.get_group(i).reset_index()
    get_price.drop('index', axis=1, inplace=True)
    groups.append(get_price)
price_vertical = pd.concat(groups, ignore_index=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# 2022년 1월 1일 기준 (유아식품 때문)
상반기_가격대 = price_vertical[price_vertical['날짜'] == '2022-01-01']

In [ ]:
# 0값은 바로 윗 행의 값으로 채우기
상반기_가격대['price'] = 상반기_가격대['price'].replace(0, method='ffill')

In [ ]:
# 가격(price) 값을 수정하는 함수 정의
def adjust_price(value):
    if value < 100:  # 십 단위
        if value % 10 >= 5:  # 일의자리 숫자가 5 이상인 경우
            value = (value // 10 + 1) * 10  # 올림 처리
        else:
            value = value // 10 * 10  # 내림 처리 (반올림)
    elif 100 <= value < 1000:  # 백 단위
        value = value // 100 * 100  # 십의자리까지 수를 0으로 처리
    elif 1000 <= value < 10000:  # 천 단위
        if (value // 100) % 10 in [8, 9]:  # 백의자리 숫자가 8 혹은 9인 경우
            value = (value // 1000 + 1) * 1000  # 올림 처리
        else:
            value = value // 100 * 100  # 십의자리까지 수를 0으로 처리
    elif 10000 <= value < 100000:  # 만 단위
        if (value // 1000) % 10 in [8, 9]:  # 천의자리 숫자가 8 혹은 9인 경우
            value = (value // 10000 + 1) * 10000  # 올림 처리
        else:
            value = value // 1000 * 1000  # 백의자리까지 수를 0으로 처리
    elif value >= 100000:  # 십만 단위 이상
        value = value // 10000 * 10000  # 천의자리까지 수를 0으로 처리
    return value


# 'price' 컬럼에 함수 적용
상반기_가격대['price'] = 상반기_가격대['price'].apply(adjust_price)

In [ ]:
# 가격 천,만,십만 단위만 남도록 수정
def adjust_price_2(value):
    # 천 단위의 숫자 조건
    if 1000 <= value < 10000:
        if (value // 100) % 10 in [8, 9]:  # 백의자리 숫자가 8 혹은 9인 경우
            value = (value // 1000 + 1) * 1000  # 올림 처리
        else:
            value = value // 1000 * 1000  # 백의자리까지 수를 0으로 처리

    # 만 단위의 숫자 조건
    elif 10000 <= value < 100000:
        if (value // 1000) % 10 in [8, 9]:  # 천의자리 숫자가 8 혹은 9인 경우
            value = (value // 10000 + 1) * 10000  # 올림 처리
        else:
            value = value // 10000 * 10000  # 천의자리까지 수를 0으로 처리

    # 십만 단위의 숫자 조건
    elif 100000 <= value < 1000000:
        if (value // 1000) % 10 in [8, 9]:  # 천의자리 숫자가 8 혹은 9인 경우
            value = (value // 100000 + 1) * 100000  # 올림 처리
        else:
           value = value // 100000 * 100000  # 천의자리까지 수를 0으로 처리

    return value


상반기_가격대['price'] = 상반기_가격대['price'].apply(adjust_price_2)

In [ ]:
상반기_가격대 = 상반기_가격대[['제품', '브랜드', 'price']].reset_index().drop('index', axis=1)

In [ ]:
상반기_가격대['price'].unique()

array([ 10000,  30000,   4000,   5000,   7000,  20000,  60000,   8000,
         3000,   2000,   1000,  50000,   6000,   9000,    500,  40000,
          700,  80000,  70000,    800,    600, 100000,  90000,    900,
          300,    100,    400, 300000, 200000, 600000, 500000, 400000,
          200,     20,     40,     70,     90,     50,     60,     80,
           30])

In [ ]:
상반기_가격대

,제품,브랜드,price
0,B002-00001-00001,B002-00001,10000
1,B002-00002-00001,B002-00002,30000
2,B002-00002-00002,B002-00002,10000
3,B002-00002-00003,B002-00002,4000
4,B002-00003-00001,B002-00003,5000
...,...,...,...
15885,B002-03799-00002,B002-03799,2000
15886,B002-03799-00003,B002-03799,20000
15887,B002-03799-00004,B002-03799,10000
15888,B002-03799-00005,B002-03799,10000


In [ ]:
상반기_가격대['price_rank'] = 상반기_가격대['price'].rank(ascending=False, method='dense')

In [ ]:
상반기_가격대['price_rank'] = 상반기_가격대['price_rank'].astype(int)

In [ ]:
상반기_가격대 = 상반기_가격대[['제품', 'price_rank']]

In [ ]:
# 빈 데이터프레임 생성
empty_data = train.copy()
empty_data.iloc[:, 6:] = 0

In [ ]:
empty_data['price_rank'] = 0

In [ ]:
col = empty_data.pop('price_rank')
empty_data.insert(6, 'price_rank', col)

In [ ]:
price_rank_map = 상반기_가격대.set_index('제품')['price_rank'].to_dict()
empty_data['price_rank'] = empty_data['제품'].map(price_rank_map).combine_first(empty_data['price_rank'])

In [ ]:
empty_data.drop(['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True)

In [ ]:
empty_data

,price_rank,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,23,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15886,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15887,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15888,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
가격대_data = empty_data.copy()
for i in tqdm(range(len(가격대_data))):
  가격대_data.iloc[i, 1:] = 가격대_data.iloc[i, 0]
가격대_data.drop('price_rank', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
가격대_data

,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,2022-01-25,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
1,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,...,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13,13
2,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
3,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,...,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21,21
4,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,...,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,...,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23,23
15886,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,...,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14
15887,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
15888,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,...,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15


In [ ]:
# 가격대_data.to_csv('./가격대_data.csv', index=False)

# **연관키워드_언급량**

In [ ]:
id_vars = ['브랜드']
brand_keyword_cnt_melted = pd.melt(brand_keyword_cnt, id_vars=id_vars, var_name='날짜', value_name='연관키워드_언급량')

# 날짜 컬럼을 datetime 타입으로 변환합니다.
brand_keyword_cnt_melted['날짜'] = pd.to_datetime(brand_keyword_cnt_melted['날짜'])

# 판매량을 날짜로 그룹화하고, 판매량을 합산하여 정리합니다.
brand_keyword_cnt_grouped = brand_keyword_cnt_melted.groupby(['브랜드', '날짜'])['연관키워드_언급량'].sum().reset_index()

In [ ]:
brand_keyword_cnt_grouped

,브랜드,날짜,연관키워드_언급량
0,B002-00001,2022-01-01,0.84131
1,B002-00001,2022-01-02,0.91383
2,B002-00001,2022-01-03,1.45053
3,B002-00001,2022-01-04,2.42239
4,B002-00001,2022-01-05,1.87119
...,...,...,...
1455025,B002-03799,2023-03-31,5.51203
1455026,B002-03799,2023-04-01,3.52480
1455027,B002-03799,2023-04-02,4.03249
1455028,B002-03799,2023-04-03,5.88917


In [ ]:
## 브랜드_검색량 와이드 포맷 변환

# 필요한 데이터 추출
brand_search_data = brand_keyword_cnt_grouped

# 날짜를 행 인덱스로 변환
brand_search_data.set_index('날짜', inplace=True)

# 피벗하여 원하는 형태로 변환
brand_search_pivoted = brand_search_data.pivot_table(index=['브랜드'], columns=brand_search_data.index, values='연관키워드_언급량', aggfunc='sum', fill_value=0)

# 세로형 날짜 정리
brand_search_pivoted.columns = pd.to_datetime(brand_search_pivoted.columns).strftime('%Y-%m-%d')

# 인덱스 위치 글자 삭제
brand_search_pivoted.columns.name = None

# 필요에 따라 인덱스를 리셋
brand_search_pivoted.reset_index(inplace=True)

In [ ]:
brand_search_pivoted

,브랜드,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,B002-00001,0.84131,0.91383,1.450530,2.422390,1.871190,1.581080,1.232950,1.174930,1.145920,1.232950,1.17493,1.102400,1.290970,1.015370,0.623730,0.652740,1.203940,0.739770,1.000870,1.276470,1.073390,0.696250,0.783280,1.145920,...,0.30461,0.203070,0.377130,0.522190,0.536690,0.420650,0.304610,0.26109,0.203070,0.391640,0.304610,0.362630,0.449660,0.319110,0.174060,0.319110,0.391640,0.377130,0.49318,0.072520,0.29010,0.31911,0.232080,0.333620,0.44966
1,B002-00002,12.64868,20.27850,15.332170,12.750210,13.562510,13.707570,11.937910,15.564250,14.084710,16.231500,14.37481,14.345800,13.388450,10.298810,14.360310,16.115460,13.881630,12.199010,13.156360,15.506230,12.808230,15.506230,17.551490,14.780960,...,9.82013,11.778350,13.968660,17.711050,15.245140,11.589780,8.587170,9.55903,10.821000,12.039450,11.343190,10.675950,9.413980,8.775740,8.920800,10.269790,11.966920,10.646930,10.41485,10.487380,9.48651,9.28343,10.429350,11.154620,11.38671
2,B002-00003,0.33362,0.43516,0.362630,0.174060,0.217580,0.464170,0.420650,0.290100,0.377130,0.754270,0.40615,0.319110,0.406150,0.435160,0.261090,0.478670,0.391640,0.275600,0.304610,1.087900,0.812300,0.870320,0.449660,0.449660,...,0.14505,0.493180,0.507680,0.652740,0.435160,0.638230,0.406150,0.39164,0.420650,0.536690,0.507680,0.551200,0.551200,0.420650,0.391640,0.536690,0.696250,0.449660,0.39164,1.029880,0.49318,0.91383,0.797790,1.015370,0.88482
3,B002-00005,1.07339,1.71163,2.016240,1.914700,1.987230,2.146790,1.682620,1.378000,1.421520,2.610960,1.90020,2.045250,1.972720,1.610090,1.726130,1.958220,2.393380,1.958220,1.755140,1.987230,1.552070,1.247460,1.769650,2.523930,...,1.14592,1.842180,2.567440,2.291840,2.581950,2.785030,1.363500,1.74064,2.146790,2.901070,2.393380,2.378880,2.045250,1.653610,3.060630,2.219320,2.509420,2.872060,2.37888,2.030750,1.53756,1.34899,1.261960,2.320850,2.30635
4,B002-00006,0.00000,0.00000,0.188558,0.246574,0.246574,0.246574,0.377139,0.087012,0.261084,0.333609,0.00000,0.290103,0.232064,0.261084,0.145052,0.101522,0.333609,0.290103,0.174048,0.435155,0.072526,0.145052,0.188558,0.261084,...,0.00000,0.072526,0.261084,0.188558,0.116032,0.072526,0.087012,0.00000,0.116032,0.246574,0.261084,0.174048,0.203068,0.101522,0.116032,0.072526,0.290103,0.087012,0.00000,0.130542,0.00000,0.00000,0.072526,0.217577,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3165,B002-03794,2.32085,2.98810,3.611830,4.061500,3.669850,3.771390,3.031620,2.988100,3.133150,11.285170,14.05570,10.603420,7.180150,4.888300,3.017110,3.408760,3.263700,3.727870,3.452270,3.698860,3.147660,2.828540,2.596460,3.452270,...,1.49405,2.509420,4.540170,11.067590,3.930950,2.436900,2.291840,1.29097,1.421520,2.190310,2.770520,2.407890,1.784160,2.045250,1.987230,2.422390,2.422390,2.756010,2.32085,2.088770,1.98723,1.07339,1.929210,2.509420,1.78416
3166,B002-03795,0.14505,0.00000,0.087030,0.072520,0.087030,0.101530,0.072520,0.130540,0.116040,0.072520,0.00000,0.072520,0.000000,0.203070,0.116040,0.072520,0.116040,0.000000,0.145050,0.000000,0.000000,0.072520,0.000000,0.130540,...,0.07252,0.000000,0.145050,0.000000,0.188560,0.000000,0.072520,0.00000,0.159550,0.087030,0.072520,0.203070,0.000000,0.000000,0.116040,0.000000,0.072520,0.000000,0.10153,0.101530,0.00000,0.00000,0.000000,0.000000,0.00000
3167,B002-03796,0.00000,0.00000,0.000000,0.000000,

In [ ]:
# 빈 데이터프레임 생성
empty_data = train.copy()
empty_data.iloc[:, 6:] = 0

In [ ]:
# 먼저 '브랜드'를 기준으로 brand_search_pivoted와 empty_data를 병합합니다.
merged_df = empty_data.merge(brand_search_pivoted, on='브랜드', suffixes=('', '_from_brand_search_pivoted'))

# 병합된 데이터프레임에서 필요한 컬럼만 업데이트합니다.
for column in brand_search_pivoted.columns:
    if column in empty_data.columns and column != '브랜드':
        merged_df[column] = merged_df[column + '_from_brand_search_pivoted']

# 불필요한 '_from_brand_search_pivoted' 접미사를 가진 컬럼들을 삭제합니다.
merged_df = merged_df.drop(columns=[col for col in merged_df if col.endswith('_from_brand_search_pivoted')])

연관키워드_언급량 = merged_df

In [ ]:
# 연관키워드_언급량.to_csv('./연관키워드_언급량.csv', index=False)

In [ ]:
# 연관키워드_언급량 = pd.read_csv('./연관키워드_언급량.csv')

In [ ]:
연관키워드_언급량

,ID,제품,대분류,중분류,소분류,브랜드,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,0,B002-00001-00001,B002-C001-0002,B002-C002-0007,B002-C003-0038,B002-00001,0.84131,0.91383,1.45053,2.42239,1.87119,1.58108,1.23295,1.17493,1.14592,1.23295,1.17493,1.10240,1.29097,1.01537,0.62373,0.65274,1.20394,0.73977,1.00087,...,0.30461,0.20307,0.37713,0.52219,0.53669,0.42065,0.30461,0.26109,0.20307,0.39164,0.30461,0.36263,0.44966,0.31911,0.17406,0.31911,0.39164,0.37713,0.49318,0.07252,0.29010,0.31911,0.23208,0.33362,0.44966
1,1,B002-00002-00001,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,12.64868,20.27850,15.33217,12.75021,13.56251,13.70757,11.93791,15.56425,14.08471,16.23150,14.37481,14.34580,13.38845,10.29881,14.36031,16.11546,13.88163,12.19901,13.15636,...,9.82013,11.77835,13.96866,17.71105,15.24514,11.58978,8.58717,9.55903,10.82100,12.03945,11.34319,10.67595,9.41398,8.77574,8.92080,10.26979,11.96692,10.64693,10.41485,10.48738,9.48651,9.28343,10.42935,11.15462,11.38671
2,2,B002-00002-00002,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,12.64868,20.27850,15.33217,12.75021,13.56251,13.70757,11.93791,15.56425,14.08471,16.23150,14.37481,14.34580,13.38845,10.29881,14.36031,16.11546,13.88163,12.19901,13.15636,...,9.82013,11.77835,13.96866,17.71105,15.24514,11.58978,8.58717,9.55903,10.82100,12.03945,11.34319,10.67595,9.41398,8.77574,8.92080,10.26979,11.96692,10.64693,10.41485,10.48738,9.48651,9.28343,10.42935,11.15462,11.38671
3,3,B002-00002-00003,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,12.64868,20.27850,15.33217,12.75021,13.56251,13.70757,11.93791,15.56425,14.08471,16.23150,14.37481,14.34580,13.38845,10.29881,14.36031,16.11546,13.88163,12.19901,13.15636,...,9.82013,11.77835,13.96866,17.71105,15.24514,11.58978,8.58717,9.55903,10.82100,12.03945,11.34319,10.67595,9.41398,8.77574,8.92080,10.26979,11.96692,10.64693,10.41485,10.48738,9.48651,9.28343,10.42935,11.15462,11.38671
4,4,B002-00003-00001,B002-C001-0001,B002-C002-0001,B002-C003-0003,B002-00003,0.33362,0.43516,0.36263,0.17406,0.21758,0.46417,0.42065,0.29010,0.37713,0.75427,0.40615,0.31911,0.40615,0.43516,0.26109,0.47867,0.39164,0.27560,0.30461,...,0.14505,0.49318,0.50768,0.65274,0.43516,0.63823,0.40615,0.39164,0.42065,0.53669,0.50768,0.55120,0.55120,0.42065,0.39164,0.53669,0.69625,0.44966,0.39164,1.02988,0.49318,0.91383,0.79779,1.01537,0.88482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,15885,B002-03799-00002,B002-C001-0003,B002-C002-0008,B002-C003-0042,B002-03799,4.55468,5.54105,6.15027,6.39686,7.00609,6.65796,5.80214,5.48302,6.03423,7.13664,7.18015,6.96257,6.94807,6.55642,6.46939,5.51203,7.74586,8.50014,9.45749,...,3.48128,4.36611,5.52654,5.27995,5.26544,4.75776,4.25007,4.04699,4.81578,5.96170,5.52654,4.98984,5.06237,5.43951,5.29445,5.10588,6.67246,6.44038,5.90368,4.93182,5.51203,3.52480,4.03249,5.88917,5.07687
15886,15886,B002-03799-00003,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-03799,4.55468,5.54105,6.15027,6.39686,7.00609,6.65796,5.80214,5.48302,6.03423,7.13664,7.18015,6.96257,6.94807,6.55642,6.46939,5.51203,7.74586,8.50014,9.45749,...,3.48128,4.36611,5.52654,5.27995,5.26544,4.75776,4.25007,4.04699,4.81578,5.96170,5.52654,4.98984,5.06237,5.43951,5.29445,5.10588,6.67246,6.44038,5.90368,4.93182,5.51203,3.52480,4.03249,5.88917,5.07687
15887,15887,B002-03799-00004,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-03799,4.55468,5.54105,6.15027,6.39686,7.00609,6

In [ ]:
brand_keyword_cnt

,브랜드,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,B002-00001,0.84131,0.91383,1.450530,2.422390,1.871190,1.581080,1.232950,1.174930,1.145920,1.232950,1.17493,1.102400,1.290970,1.015370,0.623730,0.652740,1.203940,0.739770,1.000870,1.276470,1.073390,0.696250,0.783280,1.145920,...,0.30461,0.203070,0.377130,0.522190,0.536690,0.420650,0.304610,0.26109,0.203070,0.391640,0.304610,0.362630,0.449660,0.319110,0.174060,0.319110,0.391640,0.377130,0.49318,0.072520,0.29010,0.31911,0.232080,0.333620,0.44966
1,B002-00002,12.64868,20.27850,15.332170,12.750210,13.562510,13.707570,11.937910,15.564250,14.084710,16.231500,14.37481,14.345800,13.388450,10.298810,14.360310,16.115460,13.881630,12.199010,13.156360,15.506230,12.808230,15.506230,17.551490,14.780960,...,9.82013,11.778350,13.968660,17.711050,15.245140,11.589780,8.587170,9.55903,10.821000,12.039450,11.343190,10.675950,9.413980,8.775740,8.920800,10.269790,11.966920,10.646930,10.41485,10.487380,9.48651,9.28343,10.429350,11.154620,11.38671
2,B002-00003,0.33362,0.43516,0.362630,0.174060,0.217580,0.464170,0.420650,0.290100,0.377130,0.754270,0.40615,0.319110,0.406150,0.435160,0.261090,0.478670,0.391640,0.275600,0.304610,1.087900,0.812300,0.870320,0.449660,0.449660,...,0.14505,0.493180,0.507680,0.652740,0.435160,0.638230,0.406150,0.39164,0.420650,0.536690,0.507680,0.551200,0.551200,0.420650,0.391640,0.536690,0.696250,0.449660,0.39164,1.029880,0.49318,0.91383,0.797790,1.015370,0.88482
3,B002-00005,1.07339,1.71163,2.016240,1.914700,1.987230,2.146790,1.682620,1.378000,1.421520,2.610960,1.90020,2.045250,1.972720,1.610090,1.726130,1.958220,2.393380,1.958220,1.755140,1.987230,1.552070,1.247460,1.769650,2.523930,...,1.14592,1.842180,2.567440,2.291840,2.581950,2.785030,1.363500,1.74064,2.146790,2.901070,2.393380,2.378880,2.045250,1.653610,3.060630,2.219320,2.509420,2.872060,2.37888,2.030750,1.53756,1.34899,1.261960,2.320850,2.30635
4,B002-00006,0.00000,0.00000,0.188558,0.246574,0.246574,0.246574,0.377139,0.087012,0.261084,0.333609,0.00000,0.290103,0.232064,0.261084,0.145052,0.101522,0.333609,0.290103,0.174048,0.435155,0.072526,0.145052,0.188558,0.261084,...,0.00000,0.072526,0.261084,0.188558,0.116032,0.072526,0.087012,0.00000,0.116032,0.246574,0.261084,0.174048,0.203068,0.101522,0.116032,0.072526,0.290103,0.087012,0.00000,0.130542,0.00000,0.00000,0.072526,0.217577,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3165,B002-03794,2.32085,2.98810,3.611830,4.061500,3.669850,3.771390,3.031620,2.988100,3.133150,11.285170,14.05570,10.603420,7.180150,4.888300,3.017110,3.408760,3.263700,3.727870,3.452270,3.698860,3.147660,2.828540,2.596460,3.452270,...,1.49405,2.509420,4.540170,11.067590,3.930950,2.436900,2.291840,1.29097,1.421520,2.190310,2.770520,2.407890,1.784160,2.045250,1.987230,2.422390,2.422390,2.756010,2.32085,2.088770,1.98723,1.07339,1.929210,2.509420,1.78416
3166,B002-03795,0.14505,0.00000,0.087030,0.072520,0.087030,0.101530,0.072520,0.130540,0.116040,0.072520,0.00000,0.072520,0.000000,0.203070,0.116040,0.072520,0.116040,0.000000,0.145050,0.000000,0.000000,0.072520,0.000000,0.130540,...,0.07252,0.000000,0.145050,0.000000,0.188560,0.000000,0.072520,0.00000,0.159550,0.087030,0.072520,0.203070,0.000000,0.000000,0.116040,0.000000,0.072520,0.000000,0.10153,0.101530,0.00000,0.00000,0.000000,0.000000,0.00000
3167,B002-03796,0.00000,0.00000,0.000000,0.000000,

# **연관 키워드 언급량 변화율**

In [ ]:
연관키워드_언급량 = pd.read_csv('./연관키워드_언급량.csv')

In [ ]:
연관키워드_언급량

,ID,제품,대분류,중분류,소분류,브랜드,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,0,B002-00001-00001,B002-C001-0002,B002-C002-0007,B002-C003-0038,B002-00001,0.84131,0.91383,1.45053,2.42239,1.87119,1.58108,1.23295,1.17493,1.14592,1.23295,1.17493,1.10240,1.29097,1.01537,0.62373,0.65274,1.20394,0.73977,1.00087,...,0.30461,0.20307,0.37713,0.52219,0.53669,0.42065,0.30461,0.26109,0.20307,0.39164,0.30461,0.36263,0.44966,0.31911,0.17406,0.31911,0.39164,0.37713,0.49318,0.07252,0.29010,0.31911,0.23208,0.33362,0.44966
1,1,B002-00002-00001,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,12.64868,20.27850,15.33217,12.75021,13.56251,13.70757,11.93791,15.56425,14.08471,16.23150,14.37481,14.34580,13.38845,10.29881,14.36031,16.11546,13.88163,12.19901,13.15636,...,9.82013,11.77835,13.96866,17.71105,15.24514,11.58978,8.58717,9.55903,10.82100,12.03945,11.34319,10.67595,9.41398,8.77574,8.92080,10.26979,11.96692,10.64693,10.41485,10.48738,9.48651,9.28343,10.42935,11.15462,11.38671
2,2,B002-00002-00002,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,12.64868,20.27850,15.33217,12.75021,13.56251,13.70757,11.93791,15.56425,14.08471,16.23150,14.37481,14.34580,13.38845,10.29881,14.36031,16.11546,13.88163,12.19901,13.15636,...,9.82013,11.77835,13.96866,17.71105,15.24514,11.58978,8.58717,9.55903,10.82100,12.03945,11.34319,10.67595,9.41398,8.77574,8.92080,10.26979,11.96692,10.64693,10.41485,10.48738,9.48651,9.28343,10.42935,11.15462,11.38671
3,3,B002-00002-00003,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,12.64868,20.27850,15.33217,12.75021,13.56251,13.70757,11.93791,15.56425,14.08471,16.23150,14.37481,14.34580,13.38845,10.29881,14.36031,16.11546,13.88163,12.19901,13.15636,...,9.82013,11.77835,13.96866,17.71105,15.24514,11.58978,8.58717,9.55903,10.82100,12.03945,11.34319,10.67595,9.41398,8.77574,8.92080,10.26979,11.96692,10.64693,10.41485,10.48738,9.48651,9.28343,10.42935,11.15462,11.38671
4,4,B002-00003-00001,B002-C001-0001,B002-C002-0001,B002-C003-0003,B002-00003,0.33362,0.43516,0.36263,0.17406,0.21758,0.46417,0.42065,0.29010,0.37713,0.75427,0.40615,0.31911,0.40615,0.43516,0.26109,0.47867,0.39164,0.27560,0.30461,...,0.14505,0.49318,0.50768,0.65274,0.43516,0.63823,0.40615,0.39164,0.42065,0.53669,0.50768,0.55120,0.55120,0.42065,0.39164,0.53669,0.69625,0.44966,0.39164,1.02988,0.49318,0.91383,0.79779,1.01537,0.88482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,15885,B002-03799-00002,B002-C001-0003,B002-C002-0008,B002-C003-0042,B002-03799,4.55468,5.54105,6.15027,6.39686,7.00609,6.65796,5.80214,5.48302,6.03423,7.13664,7.18015,6.96257,6.94807,6.55642,6.46939,5.51203,7.74586,8.50014,9.45749,...,3.48128,4.36611,5.52654,5.27995,5.26544,4.75776,4.25007,4.04699,4.81578,5.96170,5.52654,4.98984,5.06237,5.43951,5.29445,5.10588,6.67246,6.44038,5.90368,4.93182,5.51203,3.52480,4.03249,5.88917,5.07687
15886,15886,B002-03799-00003,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-03799,4.55468,5.54105,6.15027,6.39686,7.00609,6.65796,5.80214,5.48302,6.03423,7.13664,7.18015,6.96257,6.94807,6.55642,6.46939,5.51203,7.74586,8.50014,9.45749,...,3.48128,4.36611,5.52654,5.27995,5.26544,4.75776,4.25007,4.04699,4.81578,5.96170,5.52654,4.98984,5.06237,5.43951,5.29445,5.10588,6.67246,6.44038,5.90368,4.93182,5.51203,3.52480,4.03249,5.88917,5.07687
15887,15887,B002-03799-00004,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-03799,4.55468,5.54105,6.15027,6.39686,7.00609,6

In [ ]:
def fix_form(df):
    df['ID'] = train['ID']
    df['제품'] = train['제품']
    df['대분류'] = train['대분류']
    df['중분류'] = train['중분류']
    df['소분류'] = train['소분류']
    df['브랜드'] = train['브랜드']

    selected_columns = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
    selected_df = df[selected_columns]

    other_columns = [col for col in df.columns if col not in selected_columns]
    df = pd.concat([selected_df, df[other_columns]], axis=1)

    return df

In [ ]:
# 연관키워드_언급량 변화율
연관키워드_언급량_data = 연관키워드_언급량.drop(columns=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'])

연관키워드_언급량_change_rate = 연관키워드_언급량_data.copy()

for i in range(1, len(연관키워드_언급량_change_rate.columns)):
    prev_col = 연관키워드_언급량_change_rate.columns[i - 1]
    curr_col = 연관키워드_언급량_change_rate.columns[i]
    연관키워드_언급량_change_rate[curr_col] = (연관키워드_언급량_data[curr_col] - 연관키워드_언급량_data[prev_col]) / 연관키워드_언급량_data[prev_col]

# inf => NaN
연관키워드_언급량_change_rate = 연관키워드_언급량_change_rate.replace([np.inf, -np.inf], np.nan)

# NaN => 0
연관키워드_언급량_change_rate.fillna(0, inplace=True)

# 연관키워드_언급량_change_rate

연관키워드_언급량_change_rate = fix_form(연관키워드_언급량_change_rate)

# 2022년 1월 1일 데이터를 2023년 1월 1일 데이터로 인위적으로 대체
연관키워드_언급량_change_rate.iloc[:, 6:7] = 연관키워드_언급량_change_rate.iloc[:, 371:372]

In [ ]:
# 데이터프레임에서 무한대 값을 찾기
inf_indices = 연관키워드_언급량_change_rate[연관키워드_언급량_change_rate == np.inf].stack().index

# 무한대 값을 포함하는 행과 열 출력
if inf_indices.empty:
    print("출력할 게 없습니다.")
else:
    for row, col in inf_indices:
        print(f"Row: {row}, Column: {col}, Value: {연관키워드_언급량_change_rate.loc[row, col]}")

출력할 게 없습니다.


In [ ]:
# 연관키워드_언급량_change_rate.to_csv('./연관키워드_언급량_change_rate.csv', index=False)

In [ ]:
연관키워드_언급량_change_rate = pd.read_csv('./연관키워드_언급량_change_rate.csv')

In [ ]:
연관키워드_언급량_change_rate

,ID,제품,대분류,중분류,소분류,브랜드,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,0,B002-00001-00001,B002-C001-0002,B002-C002-0007,B002-C003-0038,B002-00001,0.000000,0.086199,0.587308,0.670003,-0.227544,-0.155040,-0.220185,-0.047058,-0.024691,0.075948,-0.047058,-0.061731,0.171054,-0.213483,-0.385712,0.046511,0.844440,-0.385542,0.352948,...,-0.159998,-0.333344,0.857143,0.384642,0.027768,-0.216214,-0.275859,-0.142871,-0.222222,0.928596,-0.222219,0.190473,0.239997,-0.290330,-0.454545,0.833333,0.227288,-0.037049,0.307719,-0.852954,3.000276,0.100000,-0.272727,0.437522,0.347821
1,1,B002-00002-00001,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,3.101209,0.603211,-0.243920,-0.168401,0.063709,0.010696,-0.129101,0.303767,-0.095060,0.152420,-0.114388,-0.002018,-0.066734,-0.230769,0.394366,0.122222,-0.138614,-0.121212,0.078478,...,-0.238470,0.199409,0.185961,0.267913,-0.139230,-0.239772,-0.259074,0.113176,0.132019,0.112600,-0.057832,-0.058823,-0.118207,-0.067797,0.016530,0.151219,0.165255,-0.110303,-0.021798,0.006964,-0.095436,-0.021407,0.123437,0.069541,0.020807
2,2,B002-00002-00002,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,3.101209,0.603211,-0.243920,-0.168401,0.063709,0.010696,-0.129101,0.303767,-0.095060,0.152420,-0.114388,-0.002018,-0.066734,-0.230769,0.394366,0.122222,-0.138614,-0.121212,0.078478,...,-0.238470,0.199409,0.185961,0.267913,-0.139230,-0.239772,-0.259074,0.113176,0.132019,0.112600,-0.057832,-0.058823,-0.118207,-0.067797,0.016530,0.151219,0.165255,-0.110303,-0.021798,0.006964,-0.095436,-0.021407,0.123437,0.069541,0.020807
3,3,B002-00002-00003,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-00002,3.101209,0.603211,-0.243920,-0.168401,0.063709,0.010696,-0.129101,0.303767,-0.095060,0.152420,-0.114388,-0.002018,-0.066734,-0.230769,0.394366,0.122222,-0.138614,-0.121212,0.078478,...,-0.238470,0.199409,0.185961,0.267913,-0.139230,-0.239772,-0.259074,0.113176,0.132019,0.112600,-0.057832,-0.058823,-0.118207,-0.067797,0.016530,0.151219,0.165255,-0.110303,-0.021798,0.006964,-0.095436,-0.021407,0.123437,0.069541,0.020807
4,4,B002-00003-00001,B002-C001-0001,B002-C002-0001,B002-C003-0003,B002-00003,0.676467,0.304358,-0.166674,-0.520007,0.250029,1.133330,-0.093759,-0.310353,0.300000,1.000027,-0.461532,-0.214305,0.272759,0.071427,-0.400014,0.833352,-0.181816,-0.296293,0.105261,...,-0.500000,2.400069,0.029401,0.285731,-0.333333,0.466656,-0.363631,-0.035726,0.074073,0.275859,-0.054054,0.085723,0.000000,-0.236847,-0.068965,0.370366,0.297304,-0.354169,-0.129031,1.629660,-0.521129,0.852934,-0.126982,0.272728,-0.128574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,15885,B002-03799-00002,B002-C001-0003,B002-C002-0008,B002-C003-0042,B002-03799,-0.043289,0.216562,0.109947,0.040094,0.095239,-0.049690,-0.128541,-0.055000,0.100530,0.182693,0.006097,-0.030303,-0.002083,-0.056368,-0.013274,-0.147983,0.405264,0.097378,0.112628,...,-0.230770,0.254168,0.265781,-0.044619,-0.002748,-0.096417,-0.106708,-0.047783,0.189966,0.237951,-0.072993,-0.097113,0.014536,0.074499,-0.026668,-0.035617,0.306819,-0.034782,-0.083334,-0.164619,0.117646,-0.360526,0.144034,0.460430,-0.137931
15886,15886,B002-03799-00003,B002-C001-0003,B002-C002-0008,B002-C003-0044,B002-03799,-0.043289,0.216562,0.109947,0.040094,0.095239,-0.049690,-0.128541,-0.055000,0.100530,0.182693,0.006097,-0.030303,-0.002083,-0.056368,-0.013274,-0.147983,0.405264,0.097378,0.112628,...,-0.230770

# **개별 키워드언급량(검색량) 변화율 - 폐기**

In [ ]:
# def fix_form(df):
#     df['ID'] = train['ID']
#     df['제품'] = train['제품']
#     df['대분류'] = train['대분류']
#     df['중분류'] = train['중분류']
#     df['소분류'] = train['소분류']
#     df['브랜드'] = train['브랜드']

#     selected_columns = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
#     selected_df = df[selected_columns]

#     other_columns = [col for col in df.columns if col not in selected_columns]
#     df = pd.concat([selected_df, df[other_columns]], axis=1)

#     return df

In [ ]:
# # 개별_브랜드_검색량 변화율
# 개별_브랜드_검색량_data = 개별_브랜드_검색량.drop(columns=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'])

# 개별_브랜드_검색량_change_rate = 개별_브랜드_검색량_data.copy()

# for i in range(1, len(개별_브랜드_검색량_change_rate.columns)):
#     prev_col = 개별_브랜드_검색량_change_rate.columns[i - 1]
#     curr_col = 개별_브랜드_검색량_change_rate.columns[i]
#     개별_브랜드_검색량_change_rate[curr_col] = (개별_브랜드_검색량_data[curr_col] - 개별_브랜드_검색량_data[prev_col]) / 개별_브랜드_검색량_data[prev_col]

# # inf => NaN
# 개별_브랜드_검색량_change_rate = 개별_브랜드_검색량_change_rate.replace([np.inf, -np.inf], np.nan)

# # NaN => 0
# 개별_브랜드_검색량_change_rate.fillna(0, inplace=True)

# # 개별_브랜드_검색량_change_rate

# 개별_브랜드_검색량_change_rate = fix_form(개별_브랜드_검색량_change_rate)

# # 2022년 1월 1일 데이터를 2023년 1월 1일 데이터로 인위적으로 대체
# 개별_브랜드_검색량_change_rate.iloc[:, 6:7] = 개별_브랜드_검색량_change_rate.iloc[:, 371:372]

In [ ]:
# 개별_브랜드_검색량_change_rate

In [ ]:
# # 데이터프레임에서 무한대 값을 찾기
# inf_indices = 개별_브랜드_검색량_change_rate[개별_브랜드_검색량_change_rate == np.inf].stack().index

# # 무한대 값을 포함하는 행과 열 출력
# if inf_indices.empty:
#     print("출력할 게 없습니다.")
# else:
#     for row, col in inf_indices:
#         print(f"Row: {row}, Column: {col}, Value: {개별_브랜드_검색량_change_rate.loc[row, col]}")

출력할 게 없습니다.


In [ ]:
# 개별_브랜드_검색량_change_rate.to_csv('./개별_브랜드_검색량_change_rate.csv', index=False)

# **개별 키워드언급량(검색량) - 폐기**

In [ ]:
# id_vars = ['브랜드']
# brand_keyword_cnt_melted = pd.melt(brand_keyword_cnt, id_vars=id_vars, var_name='날짜', value_name='연관키워드_언급량')

# # 날짜 컬럼을 datetime 타입으로 변환합니다.
# brand_keyword_cnt_melted['날짜'] = pd.to_datetime(brand_keyword_cnt_melted['날짜'])

# # 판매량을 날짜로 그룹화하고, 판매량을 합산하여 정리합니다.
# brand_keyword_cnt_grouped = brand_keyword_cnt_melted.groupby(['브랜드', '날짜'])['연관키워드_언급량'].sum().reset_index()

In [ ]:
# id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
# train_melted = pd.melt(train, id_vars=id_vars, var_name='날짜', value_name='판매량')
# train_melted['날짜'] = pd.to_datetime(train_melted['날짜'])
# train_grouped = train_melted.groupby('ID')

# groups = []
# for i in tqdm(range(15889+1)):
#     get_train = train_grouped.get_group(i).reset_index()
#     get_train.drop('index', axis=1, inplace=True)
#     groups.append(get_train)
# train_vertical = pd.concat(groups, ignore_index=True)

In [ ]:
# ## 계산을 위한 준비

# # 브랜별 보유하는 소분류 속성
# 브랜드별_소분류 = train_vertical.groupby('브랜드')['소분류'].unique().reset_index(name='소분류')

# # 브랜드별 보유하는 소분류 속성의 개수
# 브랜드별_소분류['보유하는_소분류_개수'] = 브랜드별_소분류['소분류'].apply(lambda x: len(set(x)))

# # 정리
# 브랜드별_소분류_개수 = 브랜드별_소분류[['브랜드', '보유하는_소분류_개수']]

# # 계산을 위한 데이터 병합
# 브랜드_검색량 = pd.merge(brand_keyword_cnt_grouped, 브랜드별_소분류_개수, on='브랜드', how='left')

# # 순수_브랜드_검색량 계산: 브랜드별 다중으로 소분류를 보유하는 경우, 단순히 브랜드_검색량은 브랜드 계열 내의 각 소분류 제품들이 검색되어 중복되는 값에 불가하기 때문에, 나눈다.
# 브랜드_검색량['순수_브랜드_검색량'] = 브랜드_검색량['연관키워드_언급량'] / 브랜드_검색량['보유하는_소분류_개수']

In [ ]:
# ## 브랜드_검색량 와이드 포맷 변환

# # 필요한 데이터 추출
# brand_search_data = 브랜드_검색량[['브랜드', '날짜', '순수_브랜드_검색량']]

# # 날짜를 행 인덱스로 변환
# brand_search_data.set_index('날짜', inplace=True)

# # 피벗하여 원하는 형태로 변환
# brand_search_pivoted = brand_search_data.pivot_table(index=['브랜드'], columns=brand_search_data.index, values='순수_브랜드_검색량', aggfunc='sum', fill_value=0)

# # 세로형 날짜 정리
# brand_search_pivoted.columns = pd.to_datetime(brand_search_pivoted.columns).strftime('%Y-%m-%d')

# # 인덱스 위치 글자 삭제
# brand_search_pivoted.columns.name = None

# # 필요에 따라 인덱스를 리셋
# brand_search_pivoted.reset_index(inplace=True)

In [ ]:
# # 빈 데이터프레임 생성
# empty_data = train.copy()
# empty_data.iloc[:, 6:] = 0

In [ ]:
# # 먼저 '브랜드'를 기준으로 brand_search_pivoted와 empty_data를 병합합니다.
# merged_df = empty_data.merge(brand_search_pivoted, on='브랜드', suffixes=('', '_from_brand_search_pivoted'))

# # 병합된 데이터프레임에서 필요한 컬럼만 업데이트합니다.
# for column in brand_search_pivoted.columns:
#     if column in empty_data.columns and column != '브랜드':
#         merged_df[column] = merged_df[column + '_from_brand_search_pivoted']

# # 불필요한 '_from_brand_search_pivoted' 접미사를 가진 컬럼들을 삭제합니다.
# merged_df = merged_df.drop(columns=[col for col in merged_df if col.endswith('_from_brand_search_pivoted')])

# 개별_브랜드_검색량 = merged_df

In [ ]:
# 개별_브랜드_검색량.to_csv('./개별_브랜드_검색량.csv', index=False)

# **가격**

In [ ]:
# train_contents = train.drop(columns=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'])
# sales_contents = sales.drop(columns=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'])

# price = sales_contents / train_contents

# price.fillna(0, inplace=True)
# price = price.astype(int)

In [ ]:
# def fix_form(df):
#     df['ID'] = train['ID']
#     df['제품'] = train['제품']
#     df['대분류'] = train['대분류']
#     df['중분류'] = train['중분류']
#     df['소분류'] = train['소분류']
#     df['브랜드'] = train['브랜드']

#     selected_columns = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
#     selected_df = df[selected_columns]

#     other_columns = [col for col in df.columns if col not in selected_columns]
#     df = pd.concat([selected_df, df[other_columns]], axis=1)

#     return df

In [ ]:
# price = fix_form(price)

In [ ]:
# price.to_csv('./price.csv', index=False)

# **가격대 (아마 브랜드별 가격 포지션)**

In [ ]:
# from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# # Function to fill the values as per the revised conditions
# def fill_values(row):
#     start_col = 6 # 7th column (in the actual data, this should be 6)
#     fill_value = int(row[start_col]) # Convert to integer
#     if fill_value == 0:
#         # Find the first non-zero value after the starting column
#         fill_value = int(next((value for value in row[start_col:] if value > 0), 0)) # Convert to integer
#     for i in range(start_col, len(row)):
#         if row[i] == 0:
#             row[i] = fill_value
#         elif row[i] != fill_value:
#             fill_value = int(row[i]) # Convert to integer
#     return row

# # # Apply the function to each row with tqdm
# # tqdm.pandas()
# # price_filled = price.progress_apply(fill_values, axis=1)

# **대용량/소용량, 본품/리필**

In [ ]:
# B002-03254-00002

In [ ]:
id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
train_melted = pd.melt(train, id_vars=id_vars, var_name='날짜', value_name='판매량')
train_melted['날짜'] = pd.to_datetime(train_melted['날짜'])
train_grouped = train_melted.groupby('ID')

groups = []
for i in tqdm(range(15889+1)):
    get_train = train_grouped.get_group(i).reset_index()
    get_train.drop('index', axis=1, inplace=True)
    groups.append(get_train)
train_vertical = pd.concat(groups, ignore_index=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# a = train_vertical['대분류'].unique()
# print('대분류: ', a)

In [ ]:
# 대분류:  ['B002-C001-0002' 'B002-C001-0003' 'B002-C001-0001' 'B002-C001-0005', 'B002-C001-0004']

In [ ]:
b = train_vertical[train_vertical['대분류'] == 'B002-C001-0003']['중분류'].unique()
print('중분류: ', b)

중분류:  ['B002-C002-0008' 'B002-C002-0010']


In [ ]:
c = train_vertical[train_vertical['중분류'] == 'B002-C002-0010']['소분류'].unique()
print('소분류: ', c)

소분류:  ['B002-C003-0050']


In [ ]:
# list_train = train_vertical[train_vertical['소분류'] == 'B002-C003-0045']['제품'].unique().tolist()

# for product in list_train:
#     result = product_info[product_info['제품'] == product]
#     if not result.empty:
#         print(result.to_string(index=False))
#     else:
#         print(f"No information found for product: {product}")

In [ ]:
# # 전자의 데이터프레임에서 브랜드 목록 추출
# existing_brands = train_vertical['제품'].unique()

# # 후자의 데이터프레임에서 전자의 데이터프레임에 있는 브랜드만 필터링하여 새로운 데이터프레임 생성
# filtered_df2 = product_info[product_info['제품'].isin(existing_brands)]

# # '제품' 컬럼을 기준으로 정렬
# filtered_df2_sorted = filtered_df2.sort_values(by='제품')
# product_info = filtered_df2_sorted

In [ ]:
filtered_rows = product_info[product_info['제품특성'].str.contains('본품')]

filtered_rows

In [ ]:
# # '제품특성'에 '대용량'이 포함된 행을 필터링합니다.
# filtered_rows = product_info[product_info['제품특성'].str.contains('대용량')]

# # 해당 행의 '제품' 컬럼 값 리스트를 가져옵니다.
# filtered_products = filtered_rows['제품'].tolist()

# # 각 제품에 대한 '소분류' 값을 찾습니다.
# for product in filtered_products:
#     sub_categories = train_vertical[train_vertical['제품'] == product]['소분류'].unique().tolist()
#     print(f"Product: {product}")
#     for sub_category in sub_categories:
#         print(f"  Subcategory: {sub_category}")

In [ ]:
# '제품특성'에 '대용량'이 포함된 행을 필터링합니다.
filtered_rows = product_info[product_info['제품특성'].str.contains('대용량')]

# 해당 행의 '제품' 컬럼 값 리스트를 가져옵니다.
filtered_products = filtered_rows['제품'].tolist()

# 출력 헤더
print(f"{'제품':<50} {'소분류':<30}")
print("=" * 80)

# 각 제품에 대한 '소분류' 값을 찾고 출력합니다.
for product in filtered_products:
    sub_categories = train_vertical[train_vertical['제품'] == product]['소분류'].unique().tolist()
    sub_categories_str = ", ".join(sub_categories)  # 리스트를 쉼표로 구분된 문자열로 변환
    print(f"{product:<50} {sub_categories_str:<30}")

In [ ]:
train_vertical[train_vertical['소분류']=='B002-C003-0015']

# **제품 정보 train 있는 것만 추출하기**

In [ ]:
# # 전자의 데이터프레임에서 브랜드 목록 추출
# existing_brands = train['제품'].unique()

# # 후자의 데이터프레임에서 전자의 데이터프레임에 있는 브랜드만 필터링하여 새로운 데이터프레임 생성
# filtered_df2 = product_info[product_info['제품'].isin(existing_brands)]

# # '제품' 컬럼을 기준으로 정렬
# filtered_df2_sorted = filtered_df2.sort_values(by='제품')
# product_info = filtered_df2_sorted

In [ ]:
# product_info

# **판매량변화율**

In [ ]:
# # 판매량 데이터 추출 (날짜 열만 제외하고 추출)
# train_data = train.drop(columns=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'])

# # (변화된 판매량 - 기존 판매량) / 기존 판매량에서 분모를 0으로 만들지 않기 위해 epsilon 삽입 (백분율을 생각했을 때 0.01이 적합)
# train_data = train_data.applymap(lambda x: 0.01 if x == 0 else x)

# # 판매량 변화율 계산을 위해 데이터프레임 복사
# train_change_rate = train_data.copy()

# # 판매량 변화율 계산
# for i in range(1, len(train_change_rate.columns)):
#     prev_col = train_change_rate.columns[i - 1]
#     curr_col = train_change_rate.columns[i]
#     train_change_rate[curr_col] = (train_data[curr_col] - train_data[prev_col]) / train_data[prev_col]

# # inf => NaN
# train_change_rate = train_change_rate.replace([np.inf, -np.inf], np.nan)

# # NaN => 0
# train_change_rate.fillna(0, inplace=True)

# # 판매량 변화율 데이터프레임 출력
# # train_change_rate

# train_change_rate = fix_form(train_change_rate)

# # 2022년 1월 1일 데이터를 2022년 1월 3일 데이터로 인위적으로 대체
# train_change_rate.iloc[:, 6:7] = train_change_rate.iloc[:, 8:9]

In [ ]:
# train_change_rate.to_csv('./train_change_rate.csv', index=False)

# **가격변화율**

In [ ]:
# # Function to fill the values as per the revised conditions
# def fill_values(row):
#     start_col = 6 # 7th column (in the actual data, this should be 6)
#     fill_value = int(row[start_col]) # Convert to integer
#     if fill_value == 0:
#         # Find the first non-zero value after the starting column
#         fill_value = int(next((value for value in row[start_col:] if value > 0), 0)) # Convert to integer
#     for i in range(start_col, len(row)):
#         if row[i] == 0:
#             row[i] = fill_value
#         elif row[i] != fill_value:
#             fill_value = int(row[i]) # Convert to integer
#     return row

# # Apply the function to each row with tqdm
# tqdm.pandas()
# price = price.progress_apply(fill_values, axis=1)

In [ ]:
# # 가격변화율
# price_data = price.drop(columns=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'])

# price_change_rate = price_data.copy()

# for i in range(1, len(price_change_rate.columns)):
#     prev_col = price_change_rate.columns[i - 1]
#     curr_col = price_change_rate.columns[i]
#     price_change_rate[curr_col] = (price_data[curr_col] - price_data[prev_col]) / price_data[prev_col]

# # inf => NaN
# price_change_rate = price_change_rate.replace([np.inf, -np.inf], np.nan)

# # NaN => 0
# price_change_rate.fillna(0, inplace=True)

# # price_change_rate

# price_change_rate = fix_form(price_change_rate)

# # 2022년 1월 1일 데이터를 2023년 1월 1일 데이터로 인위적으로 대체
# price_change_rate.iloc[:, 6:7] = price_change_rate.iloc[:, 371:372]

In [ ]:
# price_change_rate.to_csv('./price_change_rate.csv', index=False)

In [ ]:
# price_change_rate = pd.read_csv('./price_change_rate.csv')

# **가격탄력성 - 폐기**

In [ ]:
# price_elasticity = train_change_rate.iloc[:, 6:] /  price_change_rate.iloc[:, 6:]

# # inf => NaN
# price_elasticity = price_elasticity.replace([np.inf, -np.inf], np.nan)

# # NaN => 0
# price_elasticity.fillna(0, inplace=True)

# price_elasticity = fix_form(price_elasticity)

# # 2022년 1월 1일 데이터를 2023년 1월 2일 데이터로 인위적으로 대체
# price_elasticity.iloc[:, 6:7] = price_elasticity.iloc[:, 372:373]

In [ ]:
# price_elasticity.to_csv('./price_elasticity.csv', index=False)

# **날짜**

In [ ]:
# id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
# train_melted = pd.melt(train, id_vars=id_vars, var_name='날짜', value_name='판매량')
# train_melted['날짜'] = pd.to_datetime(train_melted['날짜'])
# train_grouped = train_melted.groupby('ID')

# groups = []
# for i in tqdm(range(15889+1)):
#     get_train = train_grouped.get_group(i).reset_index()
#     get_train.drop('index', axis=1, inplace=True)
#     groups.append(get_train)
# train = pd.concat(groups, ignore_index=True)

In [ ]:
# # 날짜와 시간 분할
# train['날짜'] = pd.to_datetime(train['날짜'])
# train['year'] = train['날짜'].dt.year
# train['day'] = train['날짜'].dt.day
# train['month'] = train['날짜'].dt.month
# train['weekday'] = train['날짜'].dt.weekday # 0-월요일

# # Create a 'holiday' column with default value 0
# train['holiday'] = 0

# holidays = [
#     ('2022-01-01', '2022-01-01', '새해'),
#     ('2022-01-31', '2022-02-02', '설날'),
#     ('2022-03-01', '2022-03-01', '삼일절'),
#     ('2022-03-09', '2022-03-09', '20대 대통령선거'),
#     ('2022-05-05', '2022-05-05', '어린이날'),
#     ('2022-05-08', '2022-05-08', '부처님 오신날'),
#     ('2022-06-01', '2022-06-01', '지방선거'),
#     ('2022-06-06', '2022-06-06', '현충일'),
#     ('2022-08-15', '2022-08-15', '광복절'),
#     ('2022-09-09', '2022-09-12', '추석'),
#     ('2022-10-03', '2022-10-03', '개천절'),
#     ('2022-10-09', '2022-10-10', '한글날'),
#     ('2022-12-25', '2022-12-25', '크리스마스'),
#     ('2023-01-01', '2023-01-01', '새해'),
#     ('2023-01-21', '2023-01-24', '설날 + 대체공휴일')
# ]

# # Iterate through the list of holidays and mark 'holiday' column accordingly
# for start_date, end_date, _ in holidays:
#     train.loc[(start_date <= train['날짜']) & (train['날짜'] <= end_date), 'holiday'] = 1

# # Create 'weekday' column if not already present (for demonstration purposes)
# train['weekday'] = train['날짜'].dt.weekday

# # Create 'weekday' column if not already present (for demonstration purposes)
# train['weekday'] = train['날짜'].dt.weekday

# # Update the 'holiday' column considering weekends
# train['holiday'] = train.apply(lambda x: 1 if (x['weekday'] >= 5 or x['holiday'] == 1) else 0, axis=1)

In [ ]:
# # year
# year_data = train[['날짜', 'ID',	'제품',	'대분류',	'중분류',	'소분류',	'브랜드',	'year']]

# # 날짜를 행 인덱스로 변환
# year_data.set_index('날짜', inplace=True)

# # 피벗하여 원하는 형태로 변환
# year_pivoted = year_data.pivot_table(index=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], columns=year_data.index, values='year', aggfunc='sum', fill_value=0)

# # 세로형 날짜 정리
# year_pivoted.columns = pd.to_datetime(year_pivoted.columns).strftime('%Y-%m-%d')

# # 인덱스 위치 글자 삭제
# year_pivoted.columns.name = None

# # 필요에 따라 인덱스를 리셋
# year_pivoted.reset_index(inplace=True)

In [ ]:
# # day
# day_data = train[['날짜', 'ID',	'제품',	'대분류',	'중분류',	'소분류',	'브랜드',	'day']]

# # 날짜를 행 인덱스로 변환
# day_data.set_index('날짜', inplace=True)

# # 피벗하여 원하는 형태로 변환
# day_pivoted = day_data.pivot_table(index=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], columns=day_data.index, values='day', aggfunc='sum', fill_value=0)

# # 세로형 날짜 정리
# day_pivoted.columns = pd.to_datetime(day_pivoted.columns).strftime('%Y-%m-%d')

# # 인덱스 위치 글자 삭제
# day_pivoted.columns.name = None

# # 필요에 따라 인덱스를 리셋
# day_pivoted.reset_index(inplace=True)

In [ ]:
# # month
# month_data = train[['날짜', 'ID',	'제품',	'대분류',	'중분류',	'소분류',	'브랜드',	'month']]

# # 날짜를 행 인덱스로 변환
# month_data.set_index('날짜', inplace=True)

# # 피벗하여 원하는 형태로 변환
# month_pivoted = month_data.pivot_table(index=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], columns=month_data.index, values='month', aggfunc='sum', fill_value=0)

# # 세로형 날짜 정리
# month_pivoted.columns = pd.to_datetime(month_pivoted.columns).strftime('%Y-%m-%d')

# # 인덱스 위치 글자 삭제
# month_pivoted.columns.name = None

# # 필요에 따라 인덱스를 리셋
# month_pivoted.reset_index(inplace=True)

In [ ]:
# # weekday
# weekday_data = train[['날짜', 'ID',	'제품',	'대분류',	'중분류',	'소분류',	'브랜드',	'weekday']]

# # 날짜를 행 인덱스로 변환
# weekday_data.set_index('날짜', inplace=True)

# # 피벗하여 원하는 형태로 변환
# weekday_pivoted = weekday_data.pivot_table(index=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], columns=weekday_data.index, values='weekday', aggfunc='sum', fill_value=0)

# # 세로형 날짜 정리
# weekday_pivoted.columns = pd.to_datetime(weekday_pivoted.columns).strftime('%Y-%m-%d')

# # 인덱스 위치 글자 삭제
# weekday_pivoted.columns.name = None

# # 필요에 따라 인덱스를 리셋
# weekday_pivoted.reset_index(inplace=True)

In [ ]:
# # holiday
# holiday_data = train[['날짜', 'ID',	'제품',	'대분류',	'중분류',	'소분류',	'브랜드',	'holiday']]

# # 날짜를 행 인덱스로 변환
# holiday_data.set_index('날짜', inplace=True)

# # 피벗하여 원하는 형태로 변환
# holiday_pivoted = holiday_data.pivot_table(index=['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], columns=holiday_data.index, values='holiday', aggfunc='sum', fill_value=0)

# # 세로형 날짜 정리
# holiday_pivoted.columns = pd.to_datetime(holiday_pivoted.columns).strftime('%Y-%m-%d')

# # 인덱스 위치 글자 삭제
# holiday_pivoted.columns.name = None

# # 필요에 따라 인덱스를 리셋
# holiday_pivoted.reset_index(inplace=True)

In [ ]:
# year_pivoted.to_csv('./year_data.csv', index=False)
# day_pivoted.to_csv('./day_data.csv', index=False)
# month_pivoted.to_csv('./month_data.csv', index=False)
# weekday_pivoted.to_csv('./weekday_data.csv', index=False)
# holiday_pivoted.to_csv('./holiday_data.csv', index=False)